# MedMNIST でどこまでできるか — 全結合から CNN へ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kyorin-phys/MLIntro2026/blob/main/appendix/medmnist_advanced.ipynb)

第4回の練習問題では、PneumoniaMNIST（小児の胸部X線、正常 / 肺炎）を、MNIST と同じ単純なモデル（画像を1列に並べて全結合層に入れる）で分類しました。
この付録では、同じデータを使って、モデルやデータの扱いを1つずつ変えながら、どこまで良くなるかを試します。

| 章 | 内容 |
|---|---|
| 1〜2 | 準備、データ（**検証データ**を使う） |
| 3 | 第4回と同じ全結合のモデル（出発点） |
| 4 | クラスの偏りに `class_weight` で対処する |
| 5 | **畳み込みニューラルネットワーク（CNN）** |
| 6 | 画像を大きくする（28×28 → 64×64） |
| 7 | **データ拡張**（画像を少しずつ変えて学習データを水増しする） |
| 8 | 判定のしきい値を検証データで決める |
| 9 | まとめ、さらに先へ |

評価はすべてテストデータの **正解率・感度・特異度・AUC** で行い、表にまとめて比べます。

Colab では「ランタイム」→「ランタイムのタイプを変更」で **GPU（T4 など）** を選ぶと速くなります（CPU のままでも動きます）。

## 1. 準備

In [ ]:
!pip install -q medmnist
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from medmnist import PneumoniaMNIST
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.layers import RandomRotation, RandomTranslation, RandomZoom
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import set_random_seed

## 2. データ

MedMNIST のデータは、**訓練（train）・検証（val）・テスト（test）** の3つに分かれています。

* **訓練データ**：モデルの重みを学習する
* **検証データ**：学習の途中経過を見る、エポック数やしきい値などの**設定を選ぶ**
* **テストデータ**：**最後に1回だけ**、選んだモデルの性能を測る

第4回の練習問題では簡単のためにテストデータを `validation_data` に使いましたが、テストデータを見ながら設定を選ぶと、テストデータに合わせ込んだ（実際より良く見える）結果になります。
この付録では、設定を選ぶのには検証データだけを使います。

画像は `(枚数, 高さ, 幅)` の形ですが、CNN では最後に **チャンネル**（白黒なら1、カラーなら3）の次元が必要なので、`(枚数, 高さ, 幅, 1)` にしておきます。
`size` で画像の大きさ（28, 64, 128, 224）を選べます。

In [ ]:
def load(size=28):
    """PneumoniaMNIST を読み込み、{'train': (画像, ラベル), 'val': ..., 'test': ...} を返す"""
    data = {}
    for split in ['train', 'val', 'test']:
        d = PneumoniaMNIST(split=split, download=True, size=size)
        images = d.imgs[..., np.newaxis] / 255.0 # (枚数, 高さ, 幅, 1) にして 0〜1 に正規化
        data[split] = (images, d.labels.flatten())
    return data

data28 = load(28)
for split, (x, y) in data28.items():
    print(f'{split:5s} 画像 {x.shape}  正常: {np.sum(y == 0):4d}  肺炎: {np.sum(y == 1):4d}  肺炎の割合: {y.mean():.2f}')

学習・評価を何度も繰り返すので、関数にまとめておきます。

* `train`：学習して学習曲線を描く。検証データの損失が5エポック続けて下がらなければ打ち切り、一番良かったときの重みに戻す（`EarlyStopping`）。
* `evaluate`：テストデータで正解率・感度・特異度・AUC を求めて `results` に記録し、混同行列を描く。

（Colab の matplotlib は日本語を表示できないので、図や表の見出しは英語にしています。）

In [ ]:
results = {} # モデルごとの評価結果をためておく

def train(model, data, class_weight=None, epochs=50):
    x_train, y_train = data['train']
    x_val, y_val = data['val']
    model.compile(optimizer='adam', loss=SparseCategoricalCrossentropy(), metrics=['accuracy'])
    early = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    history = model.fit(x_train, y_train, epochs=epochs, validation_data=(x_val, y_val),
                        class_weight=class_weight, callbacks=[early], verbose=0)
    print('学習したエポック数:', len(history.history['loss']))
    pd.DataFrame(history.history).plot(subplots=[('loss', 'val_loss'), ('accuracy', 'val_accuracy')],
                                       layout=(1, 2), figsize=(10, 3), xlabel='Epoch')
    plt.show()

def evaluate(name, model, data, threshold=0.5):
    x_test, y_test = data['test']
    prob = model.predict(x_test, verbose=0)[:, 1] # 肺炎である確率
    y_pred = (prob >= threshold).astype(int)
    tp, fn, fp, tn = confusion_matrix(y_test, y_pred, labels=[1, 0]).ravel()
    results[name] = {'accuracy': (tp + tn) / len(y_test), 'sensitivity': tp / (tp + fn), # 正解率, 感度
                     'specificity': tn / (tn + fp), 'AUC': roc_auc_score(y_test, prob)} # 特異度
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=[1, 0], display_labels=['pneumonia', 'normal'])
    plt.title(name)
    plt.show()
    return pd.DataFrame(results).T.round(3) # これまでの結果を表にして表示

## 3. 出発点：第4回と同じ全結合のモデル

画像を1列に並べ（`Flatten`）、全結合層（`Dense`）に入れるモデルです。

In [ ]:
set_random_seed(0)
dense = Sequential([
    Input(shape=(28, 28, 1)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(2, activation='softmax')
])
train(dense, data28)
evaluate('1. Dense', dense, data28)

**検証データとテストデータの差に注目**

学習曲線では検証データの正解率が 0.9 を超えているのに、テストデータでは感度は高いが特異度が低く（正常を肺炎と間違えることが多く）、正解率も下がります。

MedMNIST の PneumoniaMNIST は、元のデータセットの訓練用データを訓練・検証に分け、元のテスト用データをそのままテストにしています。
訓練と検証は同じところから取り出したよく似たデータですが、テストは別に集められたデータで、**撮影条件や患者の集まりが少し違う**可能性があります。
検証データで良くても、本番（別の施設・別の時期のデータ）で同じように当たるとは限らない、ということを表しています。
以下では、この差を縮められるかにも注目します。

## 4. クラスの偏りに対処する

訓練データは約3/4が肺炎なので、モデルは「肺炎」と答えがちになり、感度は高いが特異度が低くなりやすくなります。
第4回の練習問題では、訓練データを間引いて比率をそろえました。ここでは訓練データを捨てずに、`class_weight` で**少ないクラス（正常）の間違いを重く数えて**学習します。

このデータでは、`class_weight` の効果は小さいことが多いです。テストデータで正常を肺炎と間違えるのは、クラスの偏りよりも、上で見た検証とテストの差によるところが大きいようです。以降のモデルでも `class_weight` は使います。

In [ ]:
w = compute_class_weight('balanced', classes=np.array([0, 1]), y=data28['train'][1])
class_weight = dict(enumerate(w))
print(class_weight) # 正常(0)の重みが大きい

set_random_seed(0)
dense_w = Sequential([
    Input(shape=(28, 28, 1)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(2, activation='softmax')
])
train(dense_w, data28, class_weight=class_weight)
evaluate('2. Dense + class_weight', dense_w, data28)

## 5. 畳み込みニューラルネットワーク（CNN）

全結合のモデルは、最初に画像を1列に並べてしまうので、「どの画素とどの画素が隣り合っているか」という情報を使えません。

**CNN** は、小さなフィルタ（ここでは 3×3）を画像の上で少しずつずらしながら掛け合わせ（**畳み込み** `Conv2D`）、輪郭や模様のような**局所的なパターン**を取り出します。
**プーリング**（`MaxPooling2D`）で画像を縮小しながらこれを繰り返すと、浅い層では細かいパターン、深い層ではより広い範囲のパターンを捉えられます。
同じフィルタを画像全体で使い回すので、全結合に比べてパラメータ数も少なくてすみます。

`Dropout(0.5)` は、学習中に半分のニューロンをランダムに休ませて、過学習を防ぐ仕組みです。

In [ ]:
def make_cnn(size):
    return Sequential([
        Input(shape=(size, size, 1)),
        Conv2D(32, 3, activation='relu', padding='same'), # 3×3 のフィルタ32枚
        MaxPooling2D(),                                    # 縦横を半分に
        Conv2D(64, 3, activation='relu', padding='same'),
        MaxPooling2D(),
        Conv2D(128, 3, activation='relu', padding='same'),
        MaxPooling2D(),
        Flatten(),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dense(2, activation='softmax')
    ])

set_random_seed(0)
cnn28 = make_cnn(28)
cnn28.summary() # 層ごとの出力の形とパラメータ数
print('全結合のパラメータ数:', dense_w.count_params())

In [ ]:
train(cnn28, data28, class_weight=class_weight)
evaluate('3. CNN (28)', cnn28, data28)

全結合と比べると、AUC は上がることが多いですが、特異度は低いままです。

## 6. 画像を大きくする（28×28 → 64×64）

28×28 はかなり粗い画像です。`size=64` で同じ画像を 64×64 で読み込み、同じ形の CNN で学習します。

In [ ]:
data64 = load(64)

# 同じ画像を 28×28 と 64×64 で比べる
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(5):
    axes[0, i].imshow(data28['train'][0][i, :, :, 0], cmap='gray')
    axes[1, i].imshow(data64['train'][0][i, :, :, 0], cmap='gray')
    axes[0, i].set_title(f'label: {data28["train"][1][i]}')
    axes[0, i].axis('off')
    axes[1, i].axis('off')
plt.show()

In [ ]:
set_random_seed(0)
cnn64 = make_cnn(64)
train(cnn64, data64, class_weight=class_weight)
evaluate('4. CNN (64)', cnn64, data64)

意外かもしれませんが、このデータとモデルでは、64×64 にしても 28×28 よりはっきり良くなるとは限りません（乱数の種を変えると、どちらが良いかが入れ替わる）。
肺炎のように肺野全体の濃さで判断できる所見は 28×28 でもある程度残っており、また画像を大きくするとパラメータが増えて、少ない訓練データでは過学習しやすくなるためと考えられます。

## 7. データ拡張

訓練画像は約4700枚しかありません。**データ拡張**は、学習のたびに画像を少しずつ回転・移動・拡大して、「同じ画像だが少し違う見え方」を作り、学習データを水増しする方法です。
撮影のたびに患者の位置や向きが少しずつ違う、という現実のばらつきに強くなり、過学習を防ぐ効果があります。
上で見た「検証では良いのにテストでは悪い」という差を縮める効果も期待できます。

Keras では、データ拡張の層をモデルの先頭に入れるだけで使えます。これらの層は**学習中だけ**働き、予測のときは何もしません。

どんな変形をしてよいかは、データによって考える必要があります。
胸部X線では心臓が左にあるので、**左右反転は使いません**（反転すると実際にはありえない画像になる）。

In [ ]:
augment = Sequential([
    RandomRotation(0.03),          # ±0.03回転（約±10度）
    RandomTranslation(0.05, 0.05), # 上下左右に最大5%ずらす
    RandomZoom(0.1),               # 最大10%拡大・縮小
])

# 同じ1枚の画像に、データ拡張を8回かけてみる
x = data64['train'][0][:1]
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for ax in axes:
    ax.imshow(augment(x, training=True)[0, :, :, 0], cmap='gray')
    ax.axis('off')
plt.show()

In [ ]:
set_random_seed(0)
cnn64_aug = Sequential([Input(shape=(64, 64, 1)), augment, make_cnn(64)])
train(cnn64_aug, data64, class_weight=class_weight, epochs=100) # 毎回画像が変わるので、学習には時間がかかる
evaluate('5. CNN (64) + augmentation', cnn64_aug, data64)

感度をあまり下げずに、特異度が大きく上がります（乱数の種を変えて試しても、データ拡張なしより一貫して良くなる）。
ここまでの中で一番効果が大きい変更です。

## 8. 判定のしきい値を検証データで決める

ここまでは「肺炎の確率が0.5以上なら肺炎」と判定してきました。しきい値を下げれば感度が上がり、上げれば特異度が上がります（第3回）。

どのしきい値を使うかは、**検証データ**の ROC 曲線から決めます。ここでは「感度 + 特異度」が最大になる点（Youden の指標）を選びます。
実際の診療では、見逃しを減らしたいのか、偽陽性を減らしたいのかによって、選ぶべき点は変わります。

In [ ]:
x_val, y_val = data64['val']
prob_val = cnn64_aug.predict(x_val, verbose=0)[:, 1]
fpr, tpr, thresholds = roc_curve(y_val, prob_val)
best = np.argmax(tpr - fpr) # 感度 - (1 - 特異度) が最大の点
threshold = thresholds[best]
print(f'選んだしきい値: {threshold:.3f}（検証データで 感度 {tpr[best]:.3f}, 特異度 {1 - fpr[best]:.3f}）')

plt.plot(fpr, tpr)
plt.scatter(fpr[best], tpr[best], color='red', label=f'threshold = {threshold:.3f}')
plt.xlabel('1 - specificity')
plt.ylabel('sensitivity')
plt.title('ROC (validation)')
plt.legend()
plt.show()

In [ ]:
evaluate('6. 5 + threshold', cnn64_aug, data64, threshold=threshold)

検証データでは感度・特異度ともに高くなるしきい値を選んだはずですが、テストデータでは良くならず、かえって悪くなることもあります。
検証データでは正常の画像の確率は低く出るので、しきい値を0.5より下げても特異度はあまり落ちません。しかしテストデータでは正常の画像にも高めの確率が出るので、同じようにしきい値を下げると特異度がさらに下がってしまいます。

しきい値を検証データで決めるのは正しい手順ですが、それが役に立つのは**検証データが本番のデータとよく似ているとき**だけです。
実際に使う施設のデータで、しきい値や性能を確かめ直す必要があるのはこのためです。

## 9. まとめ

これまでの結果を並べます。AUC はしきい値によらないので、5 と 6 は同じ値になります。

In [ ]:
summary = pd.DataFrame(results).T
display(summary.round(3)) # 表として表示
summary.plot.bar(figsize=(10, 4), ylim=(0.5, 1.0), rot=20)
plt.legend(loc='lower right')
plt.show()

### 結果の見方

* **全結合 → CNN**：画素の位置関係を使えるようになり、AUC が上がりやすい。
* **画像を大きくする**：所見によっては有利だが、このデータでは効果がはっきりしなかった。計算は重くなる。
* **データ拡張**：このデータでは一番効果が大きく、特に特異度が上がった。現実のばらつきに強くなる。
* **しきい値**：感度と特異度のバランスを目的に合わせて変えられるが、決めるのに使うデータが本番と違うとうまくいかない。
* **検証とテストの差**：検証データの成績だけで判断すると、別に集めたデータでの性能を見誤る。

なお、テストデータは624枚しかないので、正解率の数％の違いは偶然でも起こりえます。乱数の種（`set_random_seed`）を変えて何回か試すと、結果のばらつきが分かります。

### さらに先へ

* **転移学習**：ImageNet（一般の写真）で学習済みの大きなモデル（ResNet, EfficientNet など、`tensorflow.keras.applications`）を土台にして、最後の層だけ学習し直す。画像を 128×128 や 224×224 にすると効果が出やすいが、GPU がほぼ必須になる。
* **アンサンブル**：複数のモデルの予測確率を平均する。
* [MedMNIST の論文](https://medmnist.com/) には、ResNet などを使ったときの各データセットの正解率・AUC がベンチマークとして載っているので、自分の結果と比べてみるとよい。

### 他のデータで試す

`PneumoniaMNIST` を、同じく2クラスの `BreastMNIST`（乳腺超音波）に置き換えると、ほぼ同じコードで試せます（こちらは訓練画像が約550枚とさらに少ない）。
ただし BreastMNIST のラベルは 0 = 悪性、1 = 正常・良性 なので、悪性を陽性（1）にするには、`load` の中でラベルを `1 - d.labels.flatten()` として0と1を入れ替えます（混同行列の表示名も変える）。
`medmnist.INFO['データ名']` で、ラベルの意味や利用条件（`license`）を確認してから使うこと。